In [5]:
"""
CELL 1: Endpoint Harmonization Audit
Scans the raw GEO metadata for the 8 core cohorts to identify survival times, 
time-to-death, and outcome definitions. This is required to harmonize the 
target variable to the clinical gold standard: 28-Day Mortality.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Define paths
BASE_DIR = Path("/workspace")
# FIX: Point exactly to the newly created unified metadata folder
RAW_METADATA_DIR = BASE_DIR / "data" / "raw" / "geo_metadata" 

all_cohorts = [
    'GSE185263', 'GSE26440', 'GSE54514', 'GSE65682', 
    'GSE95233', 'GSE236713', 'GSE272769', 'GSE69063'
]

print("==================================================")
print("GEO COHORT ENDPOINT AUDIT")
print("==================================================")

# Keywords to hunt for in the raw metadata columns
target_keywords = ['surv', 'death', 'mortality', 'day', 'time', 'outcome', 'status']

for cohort in all_cohorts:
    meta_file = RAW_METADATA_DIR / f"{cohort}_metadata.csv" 
    
    if meta_file.exists():
        df = pd.read_csv(meta_file)
        # Convert column names to lowercase to make searching easier
        cols = [c.lower() for c in df.columns]
        
        # Find columns that match our survival keywords
        relevant_cols = [df.columns[i] for i, c in enumerate(cols) if any(k in c for k in target_keywords)]
        
        print(f"\n[*] {cohort} (N={len(df)}):")
        if relevant_cols:
            print(f"    -> Found temporal/outcome columns: {relevant_cols}")
            # Print a few sample values from these columns so we can see the format
            sample_data = df[relevant_cols].dropna().head(3).to_dict('records')
            print(f"    -> Sample values: {sample_data}")
        else:
            print("    -> [WARNING] No standard survival/time columns found.")
    else:
        print(f"\n[*] {cohort}: [FILE NOT FOUND] Check path: {meta_file}")

print("\n==================================================")

GEO COHORT ENDPOINT AUDIT

[*] GSE185263 (N=392):
    -> Found temporal/outcome columns: ['in hospital mortality']
    -> Sample values: [{'in hospital mortality': 'Survived'}, {'in hospital mortality': 'Died'}, {'in hospital mortality': 'Survived'}]

[*] GSE26440 (N=130):
    -> Found temporal/outcome columns: ['outcome']
    -> Sample values: [{'outcome': 'Nonsurvivor'}, {'outcome': 'Survivor'}, {'outcome': 'Survivor'}]

[*] GSE54514 (N=163):
    -> Found temporal/outcome columns: ['disease status', 'group_day']
    -> Sample values: [{'disease status': 'healthy', 'group_day': 'HC_D1'}, {'disease status': 'healthy', 'group_day': 'HC_D1'}, {'disease status': 'healthy', 'group_day': 'HC_D1'}]

[*] GSE65682 (N=802):
    -> Found temporal/outcome columns: ['mortality_event_28days', 'time_to_event_28days']
    -> Sample values: [{'mortality_event_28days': 0.0, 'time_to_event_28days': 28.0}, {'mortality_event_28days': 0.0, 'time_to_event_28days': 28.0}, {'mortality_event_28days': 0.0, 'tim

In [6]:
"""
CELL 2: Target Variable Harmonization (The 28-Day Proxy)
Applies strict, cohort-specific mapping logic to unify disparate mortality columns 
(e.g., 'mort30', 'in hospital mortality', 'survival') into a single, binary 
target variable: 1 (Dead), 0 (Alive). 
Drops patients with unknown/NA outcomes to ensure absolute target fidelity 
for the LOCO external validation.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
BASE_DIR = Path("/workspace")
RAW_METADATA_DIR = BASE_DIR / "data" / "raw" / "geo_metadata"
TENSOR_DIR = BASE_DIR / "data" / "processed" / "ml_tensors"

# Ensure tensor directory exists
TENSOR_DIR.mkdir(parents=True, exist_ok=True)

all_cohorts = [
    'GSE185263', 'GSE26440', 'GSE54514', 'GSE65682', 
    'GSE95233', 'GSE236713', 'GSE272769', 'GSE69063'
]

print("[*] INITIATING STRICT ENDPOINT HARMONIZATION...")

harmonized_records = []

# ==========================================
# COHORT-SPECIFIC MAPPING LOGIC
# ==========================================
def extract_mortality(row, cohort):
    """
    Maps the custom outcome string to a strict 1 (Dead) or 0 (Alive)
    based on the exact unique values found in the harvester reports.
    """
    if cohort == 'GSE185263':
        val = str(row.get('in hospital mortality', '')).strip().lower()
        if val == 'died': return 1.0
        if val == 'survived': return 0.0
        
    elif cohort == 'GSE26440':
        val = str(row.get('outcome', '')).strip().lower()
        if val == 'nonsurvivor': return 1.0
        if val == 'survivor': return 0.0
        
    elif cohort == 'GSE54514':
        val = str(row.get('disease status', '')).strip().lower()
        if val == 'sepsis nonsurvivor': return 1.0
        if val in ['sepsis survivor', 'healthy']: return 0.0
        
    elif cohort == 'GSE65682':
        val = str(row.get('mortality_event_28days', '')).strip()
        if val in ['1', '1.0']: return 1.0
        if val in ['0', '0.0']: return 0.0
        
    elif cohort == 'GSE95233':
        val = str(row.get('survival', '')).strip().lower()
        if val == 'non survivor': return 1.0
        if val == 'survivor': return 0.0
        
    elif cohort == 'GSE236713':
        val = str(row.get('died/survived', '')).strip().lower()
        if val == 'died': return 1.0
        if val == 'survived': return 0.0
        
    elif cohort == 'GSE272769':
        val = str(row.get('mort30', '')).strip().lower()
        if val == 'yes': return 1.0
        if val == 'no': return 0.0
        
    elif cohort == 'GSE69063':
        val = str(row.get('severity', '')).strip().lower()
        if val == 'died': return 1.0
        if val in ['alive', 'mild', 'moderate', 'severe']: return 0.0

    # Catch-all for missing/unknown data ('na', 'n/a', 'unknown', missing keys)
    return np.nan 

# ==========================================
# PROCESSING LOOP
# ==========================================
for cohort in all_cohorts:
    meta_file = RAW_METADATA_DIR / f"{cohort}_metadata.csv"
    
    if meta_file.exists():
        df = pd.read_csv(meta_file)
        
        initial_count = len(df)
        mapped_count = 0
        
        for _, row in df.iterrows():
            patient_id = str(row.get('Patient_ID', '')).strip()
            mortality = extract_mortality(row, cohort)
            
            # Only append patients with a successfully mapped binary mortality outcome
            if patient_id and pd.notna(mortality):
                harmonized_records.append({
                    'Patient_ID': patient_id,
                    'Cohort': cohort,
                    'Mortality': int(mortality)
                })
                mapped_count += 1
                
        print(f"    -> {cohort:<10} | Mapped {mapped_count}/{initial_count} patients.")
    else:
        print(f"    [!] Missing metadata for {cohort}")

# ==========================================
# EXPORT MASTER LABELS
# ==========================================
df_master = pd.DataFrame(harmonized_records)
# Drop duplicates just in case the raw GEO sets had duplicate GSM IDs
df_master = df_master.drop_duplicates(subset=['Patient_ID'])

# Overwrite the master label file with the harmonized data
out_file = TENSOR_DIR / "y_master.csv"
df_master.to_csv(out_file, index=False)

print("\n==================================================")
print(f"[*] HARMONIZATION COMPLETE.")
print(f"[*] Extracted {len(df_master)} valid clinical endpoints.")
print(f"[*] Overall Sepsis Mortality Rate: {(df_master['Mortality'].mean() * 100):.1f}%")
print(f"[*] Overwritten master labels: {out_file.name}")
print("==================================================")

[*] INITIATING STRICT ENDPOINT HARMONIZATION...
    -> GSE185263  | Mapped 345/392 patients.
    -> GSE26440   | Mapped 116/130 patients.
    -> GSE54514   | Mapped 163/163 patients.
    -> GSE65682   | Mapped 479/802 patients.
    -> GSE95233   | Mapped 102/124 patients.
    -> GSE236713  | Mapped 447/447 patients.
    -> GSE272769  | Mapped 161/161 patients.
    -> GSE69063   | Mapped 132/153 patients.

[*] HARMONIZATION COMPLETE.
[*] Extracted 1945 valid clinical endpoints.
[*] Overall Sepsis Mortality Rate: 21.2%
[*] Overwritten master labels: y_master.csv


In [1]:
"""
CELL 3: Master Tensor Synchronization
Permanently synchronizes the harmonized 28-day endpoints with the master 
transcriptomic tensor directly on disk. Ensures X_master, meta_master, 
and y_master have the exact same shape and row order.
"""
import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
BASE_DIR = Path("/workspace")
TENSOR_DIR = BASE_DIR / "data" / "processed" / "ml_tensors"

print("[*] INITIATING MASTER TENSOR SYNCHRONIZATION...")

x_path = TENSOR_DIR / "X_master.csv.gz"
meta_path = TENSOR_DIR / "meta_master.csv"
y_path = TENSOR_DIR / "y_master.csv" # The one we just generated in Cell 2

print("    -> Loading tensors into memory...")
X = pd.read_csv(x_path, compression='gzip')
meta = pd.read_csv(meta_path)
y_harmonized = pd.read_csv(y_path)

# ==========================================
# ALIGNMENT LOGIC
# ==========================================
# 1. Map the harmonized labels to the existing 1,819 patients in X
label_dict = dict(zip(y_harmonized['Patient_ID'].astype(str).str.strip(), y_harmonized['Mortality']))
meta['Patient_ID'] = meta['Patient_ID'].astype(str).str.strip()

# 2. Extract aligned mortality
y_aligned = meta['Patient_ID'].map(label_dict)

# 3. Filter out patients that didn't get a valid 28-day label
valid_indices = y_aligned.notna()

X_clean = X[valid_indices].reset_index(drop=True)
meta_clean = meta[valid_indices].reset_index(drop=True)
y_clean = pd.DataFrame({
    'Patient_ID': meta_clean['Patient_ID'], 
    'Mortality': y_aligned[valid_indices].astype(int)
})

print(f"    -> Filtered out {len(X) - len(X_clean)} patients lacking valid 28-day endpoints.")

# ==========================================
# OVERWRITE AND SAVE
# ==========================================
print("    -> Overwriting tensors on disk to ensure permanent alignment...")
X_clean.to_csv(x_path, compression='gzip', index=False)
meta_clean.to_csv(meta_path, index=False)
y_clean.to_csv(y_path, index=False)

print("\n" + "="*50)
print("[*] SYNCHRONIZATION COMPLETE. ROOT CAUSE FIXED.")
print(f"    -> Final X_master shape:    {X_clean.shape}")
print(f"    -> Final y_master shape:    {y_clean.shape}")
print(f"    -> Final meta_master shape: {meta_clean.shape}")
print("="*50)

[*] INITIATING MASTER TENSOR SYNCHRONIZATION...
    -> Loading tensors into memory...
    -> Filtered out 0 patients lacking valid 28-day endpoints.
    -> Overwriting tensors on disk to ensure permanent alignment...

[*] SYNCHRONIZATION COMPLETE. ROOT CAUSE FIXED.
    -> Final X_master shape:    (1819, 7902)
    -> Final y_master shape:    (1819, 2)
    -> Final meta_master shape: (1819, 2)
